<a href="https://colab.research.google.com/github/anshulk-cmu/blackbox-nlp-2026/blob/main/notebooks/01_tokenizer_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Tokenizer audit

**Project.** Off-Manifold Failure (BlackboxNLP 2026 archival).

**Goal of this notebook.** For each of the three target models (GPT-J 6B, Pythia 6.9B, Llama 3.1 8B), determine which `(a, b) in {0,...,99}^2` pairs have:
- single-token operand `a`
- single-token operand `b`
- single-token answer `s = a + b`

Then compute the three-way intersection (pairs retained by every model) and save to Drive.

**Why this is Phase 1.** Defining the analysis dataset is a prerequisite for every later phase. See [colab_execution_plan.md section 5](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/colab_execution_plan.md) for full context, and [full_paper_plan.md section 3.2](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/full_paper_plan.md) for the methodology.

**Runtime.** CPU is sufficient (no model load needed for tokenizer ops). Estimated wall time: <= 10 minutes.

**Inputs.** None on Drive (this is the first notebook).

**Outputs to Drive.**
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/gpt-j-6b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/pythia-6.9b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/llama-3.1-8b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/intersection.json`

## 1. Standard prelude (Drive mount, repo clone, dependency install, HF login)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
PROJECT = '/content/drive/MyDrive/blackbox_nlp_2026'
CODE_DIR = f'{PROJECT}/code_repo'
REPO_URL = 'https://github.com/anshulk-cmu/blackbox-nlp-2026.git'

os.makedirs(PROJECT, exist_ok=True)
if not os.path.exists(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}
!git pull --ff-only

/content/drive/MyDrive/blackbox_nlp_2026/code_repo
Already up to date.


In [3]:
# Pin dependency versions per colab_execution_plan.md section 3.
# Phase 1 only needs transformers (for tokenizers); torch is already on Colab.
!pip install -q \
    transformers==4.45.0 \
    huggingface_hub==0.25.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 129.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.25.1 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.1 which is incompatible.


In [4]:
# HuggingFace login. HF_TOKEN must be stored in Colab Secrets (key icon in sidebar).
# DO NOT paste the token in this notebook or any committed file.
from google.colab import userdata
import huggingface_hub
hf_token = userdata.get('HF_TOKEN')
assert hf_token is not None and hf_token.startswith('hf_'), \
    'HF_TOKEN not set in Colab Secrets. Add it via the key icon in the sidebar.'
huggingface_hub.login(hf_token)
print('HF login OK')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful
HF login OK


In [5]:
# Confirm runtime (Phase 1 does not need a GPU; this is just informational).
import torch, sys
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
Device: NVIDIA L4


## 2. Load all three tokenizers

We do **not** load the model weights here -- only the tokenizers, which are tiny (a few MB each).

In [6]:
from transformers import AutoTokenizer

MODELS = {
    'gpt-j-6b':     'EleutherAI/gpt-j-6B',
    'pythia-6.9b':  'EleutherAI/pythia-6.9b',
    'llama-3.1-8b': 'meta-llama/Llama-3.1-8B',
}

tokenizers = {}
for key, hf_name in MODELS.items():
    print(f'Loading tokenizer for {key} ({hf_name})...')
    tokenizers[key] = AutoTokenizer.from_pretrained(hf_name)
    print(f'  vocab size: {tokenizers[key].vocab_size}')
print('All tokenizers loaded.')

Loading tokenizer for gpt-j-6b (EleutherAI/gpt-j-6B)...


tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  vocab size: 50254
Loading tokenizer for llama-3.1-8b (meta-llama/Llama-3.1-8B)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

  vocab size: 128000
All tokenizers loaded.


## 3. Run the audit per model

Imports `code/tokenizer_audit.py` (already cloned with the repo) and applies it to each tokenizer. The audit logic is the same code path the local self-test exercises -- see [code/tokenizer_audit.py](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/code/tokenizer_audit.py) for the implementation.

In [7]:
# Insert code/ directly (NOT the repo root) to avoid shadowing Python's
# built-in `code` module.
sys.path.insert(0, f'{CODE_DIR}/code')
from tokenizer_audit import (
    audit_tokenizer, intersect_audits, save_audit, save_intersection,
    summary_report, PROMPT_TEMPLATES,
)

AUDIT_DIR = f'{PROJECT}/tokenizer_audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

audits = []
for key in MODELS:
    print(f'\nAuditing {key}...')
    print(f'  prompt template: {PROMPT_TEMPLATES[key]!r}')
    audit = audit_tokenizer(tokenizers[key], key)
    audits.append(audit)
    out_path = f'{AUDIT_DIR}/{key}.json'
    save_audit(audit, out_path)
    print(f'  retained {audit["n_retained"]} / 10000, dropped {audit["n_dropped"]}.')
    print(f'  saved to {out_path}')


Auditing gpt-j-6b...
  prompt template: 'Output ONLY a number. {a}+{b}='
  retained 10000 / 10000, dropped 0.
  saved to /content/drive/MyDrive/blackbox_nlp_2026/tokenizer_audit/gpt-j-6b.json

Auditing pythia-6.9b...
  prompt template: 'Output ONLY a number. {a}+{b}='
  retained 10000 / 10000, dropped 0.
  saved to /content/drive/MyDrive/blackbox_nlp_2026/tokenizer_audit/pythia-6.9b.json

Auditing llama-3.1-8b...
  prompt template: 'The following is a correct addition problem.\n{a}+{b}='
  retained 0 / 10000, dropped 10000.
  saved to /content/drive/MyDrive/blackbox_nlp_2026/tokenizer_audit/llama-3.1-8b.json


## 4. Three-way intersection

In [8]:
intersection = intersect_audits(audits)
intersect_path = f'{AUDIT_DIR}/intersection.json'
save_intersection(intersection, audits, intersect_path)
print(f'Saved {len(intersection)} retained pairs to {intersect_path}')

Saved 0 retained pairs to /content/drive/MyDrive/blackbox_nlp_2026/tokenizer_audit/intersection.json


## 5. Summary report

In [9]:
print(summary_report(audits, intersection))

Tokenizer audit summary:
  gpt-j-6b      : 10000 / 10000 retained (100.0%)
  pythia-6.9b   : 10000 / 10000 retained (100.0%)
  llama-3.1-8b  :     0 / 10000 retained (0.0%)
  INTERSECTION  :     0 / 10000 retained by all (0.0%)


## 6. Sanity assertion (pre-registered gate)

Per [colab_execution_plan.md section 5](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/colab_execution_plan.md), the three-way intersection must contain at least 7000 pairs. If less, investigate before proceeding to Phase 2.

In [10]:
n_int = len(intersection)
if n_int < 7000:
    print(f'WARNING: intersection has only {n_int} pairs (target >= 7000).')
    print('Investigate per-model dropped lists before continuing to Phase 2.')
    for au in audits:
        bad_ops = [n for n, ok in au['operand_ok'].items() if not ok]
        bad_ans = [n for n, ok in au['answer_ok'].items() if not ok]
        print(f'  {au["model"]}: multi-token operands={bad_ops!r}, multi-token answers={bad_ans!r}')
else:
    print(f'PASS: intersection has {n_int} pairs (>= 7000 target).')
    print('Proceed to Phase 2 (accuracy reproduction).')

Investigate per-model dropped lists before continuing to Phase 2.
  gpt-j-6b: multi-token operands=[], multi-token answers=[]
  pythia-6.9b: multi-token operands=[], multi-token answers=[]
  llama-3.1-8b: multi-token operands=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99'], multi-token answers=[]


## 7. Optional: per-tokenizer diagnostic

If any model dropped pairs, the cell below shows which integer values in 0..198 are multi-token. This is the most common reason for drops (e.g., some BPE tokenizers split certain three-digit numbers like 100, 200).

In [11]:
for au in audits:
    bad_ans_ints = sorted(int(n) for n, ok in au['answer_ok'].items() if not ok)
    bad_op_ints = sorted(int(n) for n, ok in au['operand_ok'].items() if not ok)
    print(f'\n{au["model"]}:')
    print(f'  Multi-token operands (0..99):  {bad_op_ints if bad_op_ints else "(none)"}')
    print(f'  Multi-token answers (0..198):  {bad_ans_ints if bad_ans_ints else "(none)"}')


gpt-j-6b:
  Multi-token operands (0..99):  (none)
  Multi-token answers (0..198):  (none)

pythia-6.9b:
  Multi-token operands (0..99):  (none)
  Multi-token answers (0..198):  (none)

llama-3.1-8b:
  Multi-token operands (0..99):  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
  Multi-token answers (0..198):  (none)


## 8. Done

Phase 1 outputs are now in `/MyDrive/blackbox_nlp_2026/tokenizer_audit/`. Proceed to Phase 2 (accuracy reproduction) once:

1. The intersection size is comfortable (>= 7000).
2. The per-model retained counts roughly match KT's reported coverage.

Phase 2 lives in `notebooks/02a_accuracy_gptj.ipynb`, `02b_accuracy_pythia.ipynb`, `02c_accuracy_llama.ipynb` (to be written next).